# VL-JEPA 2 — Vision-Language JEPA · Inference

A larger, **video-capable** vision-language JEPA following the VL-JEPA paper
([arXiv:2512.10942](https://arxiv.org/pdf/2512.10942)). Frames + a text query are
encoded into a shared space; downstream tasks are solved by comparing embeddings
rather than generating tokens.

### Inference modes shown here
1. **Caption** — nearest-neighbor readout against a bank of candidate texts.
2. **Discriminative VQA** — pick the best answer among per-sample candidates by
   cosine similarity.
3. **Selective decoding** *(paper Sec. 4.6, left as an optional stub)* — segment
   the embedding stream and decode only representative points.

The notebook offers two config presets — a **tiny** model (fast smoke test) and
the **SFT** model (after supervised fine-tuning). Run one config cell, not both.


In [17]:
# Confirm GPU availability.
!nvidia-smi

Sun Jul  5 17:11:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   33C    P0             62W /  300W |   17401MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# argparse is stubbed out because we set paths directly in cells below.
#import argparse
import json
from pathlib import Path

import torch

In [4]:
# Local VL-JEPA 2 package: JSONL dataset, eval task, NN decoder, model builder.
from vl_jepa2.data.dataset import VisionLanguageJsonlDataset, vl_collate
from vl_jepa2.eval.tasks import discriminative_match
from vl_jepa2.inference.decoder import NearestNeighborDecoder
from vl_jepa2.train.build import build_model
from vl_jepa2.utils.config import load_yaml, validate_train_config

In [9]:
# PRESET A: tiny model — fastest path to verify the pipeline end-to-end.
# test with tiny model
cfg = load_yaml("./vl_jepa2/configs/inference_tiny.yaml")
model_id = "./vl_jepa2/checkpoint/step_0000002.pt"
manifest_path = "vl_jepa2_tiny_infer_manifest.jsonl"
text_bank_path = "vl_jepa2_tiny_text_bank.txt"

In [6]:
# PRESET B: SFT model — real quality. Run either this cell OR the tiny one above,
# whichever checkpoint you have; the later cells use whatever cfg/paths are set.
# test with sft model
cfg = load_yaml("./vl_jepa2/configs/inference.yaml")
model_id = "./outputs/sft/step_0000100.pt"
manifest_path = "vl_jepa2_sft_infer_manifest.jsonl"
text_bank_path = "vl_jepa2_tiny_text_bank.txt"

In [10]:
# Validate config and pick the device (falls back to CPU if no CUDA).
validate_train_config(cfg, require_runtime=True, require_train=False)
device = torch.device(cfg["runtime"]["device"] if torch.cuda.is_available() else "cpu")

In [11]:
model = build_model(cfg)

/home/sagemaker-user/rlvr_lwm/vl_jepa2/models/vljepa.py:247: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.predictor = nn.TransformerEncoder(encoder_layer, num_layers=cfg.predictor_layers)


In [12]:
# strict=False tolerates head/aux keys that differ between train and infer.
ckpt = torch.load(model_id, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=False)
model.to(device).eval()

VLJEPA(
  (x_encoder): VisionEncoder(
    (backbone): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): GELU(approximate='none')
      (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): GELU(approximate='none')
      (4): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (proj): Linear(in_features=64, out_features=64, bias=True)
  )
  (query_encoder): ToyTextEncoder(
    (embedding): Embedding(4096, 128)
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (query_embed): Embedding(4096, 128)
  (predictor): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_featu

In [13]:
# Each JSONL line points to a video/frames + a query (+ optional candidates).
ds = VisionLanguageJsonlDataset(
    manifest_path=manifest_path,
    num_frames=cfg["data"]["num_frames"],
    image_size=cfg["data"]["image_size"],
)

In [14]:
# MODE 1 (caption): embed each sample, find the closest text in the bank.
# Caption mode uses lightweight nearest-neighbor readout in text-embedding bank.
if text_bank_path is None:
    raise ValueError("--text-bank is required for caption mode.")
    
text_bank = []
with open(text_bank_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            text_bank.append(json.loads(line)["text"] if line.startswith("{") else line)
decoder = NearestNeighborDecoder(model, text_bank)
decoder.build(device)

for i in range(len(ds)):
    batch = vl_collate([ds[i]])
    with torch.autocast(device_type="cuda", dtype=torch.float):
        pred = model.predict_embedding(batch["frames"].to(device), batch["query"])
        
    text = decoder.decode(pred, topk=1)[0]
    print(json.dumps({"idx": i, "prediction": text}, ensure_ascii=False))

{"idx": 0, "prediction": "blue square"}
{"idx": 1, "prediction": "blue square"}


In [15]:
# MODE 2 (discriminative VQA): score each candidate answer, pick the best.
# Discriminative VQA chooses among provided candidates by cosine similarity.
for i in range(len(ds)):
    batch = vl_collate([ds[i]])
    if not batch["candidates"][0]:
        raise ValueError("Sample missing `candidates` for discriminative mode.")

    with torch.autocast(device_type="cuda", dtype=torch.float):
        pred = model.predict_embedding(batch["frames"].to(device), batch["query"])

    ans = discriminative_match(model, pred, [batch["candidates"][0]])[0]
    print(json.dumps({"idx": i, "prediction": ans}, ensure_ascii=False))


{"idx": 0, "prediction": "blue square"}
{"idx": 1, "prediction": "blue square"}


In [16]:
# MODE 3 (selective decoding): optional/advanced — left as a stub to try later.
# Selective decoding mode follows paper Sec. 4.6: test later 
# segment the embedding stream, decode only representative points.
"""
from vl_jepa2.inference.selective import selective_decode_points

if len(ds) != 1:
    raise ValueError("Selective mode expects a single sample video in manifest.")
batch = vl_collate([ds[0]])
frames = batch["frames"].to(device)[0]  # [T, C, H, W]
query = batch["query"][0]
per_t = []
for t in range(frames.shape[0]):
    emb = model.predict_embedding(frames[t : t + 1].unsqueeze(0), [query])[0]
    per_t.append(emb)
stream = torch.stack(per_t, dim=0)
idx, emb = selective_decode_points(stream, num_segments=args.num_segments, use_avg_pool=True)
print(json.dumps({"decode_indices": idx, "num_points": len(idx)}))
if args.text_bank is not None:
    text_bank = [x.strip() for x in Path(args.text_bank).read_text(encoding="utf-8").splitlines() if x.strip()]
    decoder = NearestNeighborDecoder(model, text_bank)
    decoder.build(device)
    texts = decoder.decode(emb, topk=1)
    print(json.dumps({"decoded_texts": texts}, ensure_ascii=False))
"""    

'\nfrom vl_jepa2.inference.selective import selective_decode_points\n\nif len(ds) != 1:\n    raise ValueError("Selective mode expects a single sample video in manifest.")\nbatch = vl_collate([ds[0]])\nframes = batch["frames"].to(device)[0]  # [T, C, H, W]\nquery = batch["query"][0]\nper_t = []\nfor t in range(frames.shape[0]):\n    emb = model.predict_embedding(frames[t : t + 1].unsqueeze(0), [query])[0]\n    per_t.append(emb)\nstream = torch.stack(per_t, dim=0)\nidx, emb = selective_decode_points(stream, num_segments=args.num_segments, use_avg_pool=True)\nprint(json.dumps({"decode_indices": idx, "num_points": len(idx)}))\nif args.text_bank is not None:\n    text_bank = [x.strip() for x in Path(args.text_bank).read_text(encoding="utf-8").splitlines() if x.strip()]\n    decoder = NearestNeighborDecoder(model, text_bank)\n    decoder.build(device)\n    texts = decoder.decode(emb, topk=1)\n    print(json.dumps({"decoded_texts": texts}, ensure_ascii=False))\n'